Kod ma na celu stworzenie, nauczenie i przetestowanie sieci w celu klasyfikacji śmieci na podstawie zdjęć. Poniżej znajduje się funkcja do ekstrakcji cech ze zdjęcia. Zwraca wektor numerycznych cech, teraz ok. 60 ale cały kod jest elastyczny, więc można dodawać co się chce tutaj. Oczywiście jak się coś doda/usunie to trzeba wytrenować nowy model, bo wtedy stary nie zadziała.

In [4]:
import cv2
import numpy as np
import torch
from skimage.feature import local_binary_pattern


def extract_features(image_path):

    img = cv2.imread(image_path)
    
    if img is None: 
        return None
    
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # CECHY KOLORU PO KOLEI

    # Histogramy HSV (H - odcień, S - nasycenie, V - jasność)
    hist_h = cv2.calcHist([img_hsv], [0], None, [16], [0, 180])
    hist_s = cv2.calcHist([img_hsv], [1], None, [16], [0, 256])
    hist_v = cv2.calcHist([img_hsv], [2], None, [16], [0, 256])

    #Normalizacja
    hist_h = hist_h / (hist_h.sum() + 1e-6)
    hist_s = hist_s / (hist_s.sum() + 1e-6)
    hist_v = hist_v / (hist_v.sum() + 1e-6)

    # Odchylenie standardowe (czy kolor jest jednolity czy taki ala poszarpany)
    std_color = cv2.meanStdDev(img_hsv)[1].flatten()

    # CECHY KSZTAŁTU - po kolei zamieniamy na szary (musi być żeby treshold działał), czarno biały i szukamy krawędzi 
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Local_binary_pattern
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
    hist = hist.astype("float") / (hist.sum() + 1e-6)

    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        # Współczynnik zwartości (Compactness)
        compactness = (4 * np.pi * area) / (perimeter**2) if perimeter > 0 else 0
        # Prostokąt otaczający (Aspect Ratio)
        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = float(w)/h

    else:
        compactness, aspect_ratio = 0, 0



    # Wektor wszystkich cech
    features = np.concatenate([
    hist_h.flatten(),
    hist_s.flatten(),
    hist_v.flatten(),
    std_color,
    hist,  # LBP
    [compactness, aspect_ratio]])

    return torch.tensor(features, dtype=torch.float32)



Poniżej znajduje się kod do tworzenia sieci neuronowej

In [5]:
import torch.nn as nn
import torch.optim as optim


class WasteNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(WasteNet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2), # Losowo wyłącza neurony, przez co teoretycznie wychodzi to na plus reszcie, ale nie wiem czy to takie skuteczne
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.Dropout(0.2),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes) # klasy: carboard, glass, metal, paper, plastic, trash
        )
    
    def forward(self, x):
        return self.fc(x)


def model_in(input_size=8, num_classes=6, learning_rate=0.001):
    # Inicjalizacja
    net = WasteNet(input_size=input_size, num_classes=num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=learning_rate)

    return net, criterion, optimizer

Poniżej kod, który służy do stworzenia i trenowania sieci, używa dwóch poprzednich funkcji. Klasa CustomDataset służy do pobrania zdjęć z folderów i odpowiedniej kategori do każdego zdjęcia. Wywołuje to raz na początku żeby było szybciej. Żeby to działało trzeba wywołać poprzednie fragementy i po chwili odpali się uczenie. Można zmienić liczbę epok w pętli. Przykładowy model jest zapisany w pliku waste_model.pth

In [7]:
import os
import json
import przerabianko
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


class CustomDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.classes = sorted(os.listdir(root_dir))
        self.samples = []

        # Robimy listę wszystkich plików raz na początku
        for target_class in self.classes:
            class_dir = os.path.join(root_dir, target_class)
            if not os.path.isdir(class_dir):
                continue
            for img_name in os.listdir(class_dir):
                self.samples.append((
                    os.path.join(class_dir, img_name),
                    self.classes.index(target_class)
                ))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        features = przerabianko.extract_features(img_path)
        
        # Obsługa błędnych zdjęć (np. jeśli OpenCV nie wczyta pliku)
        if features is None:
            return torch.zeros(self.number_features), torch.tensor(label)
            
        return features, torch.tensor(label)


if __name__ == "__main__":
    folder_path = 'Waste-Classification-1/train'
    MODEL_PATH = "waste_model_2.pth"

    dataset = CustomDataset(root_dir=folder_path)

    sample_features, _ = dataset[0]
    input_size = sample_features.shape[0] # Żeby był flex i nie trzeba by było liczyć tego co zmienioną cechę.

    dataloader = DataLoader(dataset, batch_size=4, shuffle=True) # Pakuje zdjęcia w paczki i daje w sieć na raz, co przyspiesza trenowanie
    
    model, criterion, optimizer = model_in(input_size=input_size)
    
    for epoch in range(10):
        running_loss = 0.0
        for i, data in enumerate(dataloader):
            inputs, labels = data
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels.squeeze())
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch + 1}, Loss: {running_loss / len(dataloader)}")
    print('Koniec treningu')

    # Zapisujemy same wagi (state_dict)
    torch.save(model.state_dict(), MODEL_PATH)

    # Zapis klas, tak o żeby było, w sumie nie wiem po co może później się przyda
    with open("classes.json", "w") as f:
        json.dump(dataset.classes, f)

    print(f"Model zapisany jako {MODEL_PATH}")


Epoch 1, Loss: 1.7003366488676805
Epoch 2, Loss: 1.6513590863926917
Epoch 3, Loss: 1.6256365436234625
Epoch 4, Loss: 1.569131449067215
Epoch 5, Loss: 1.4832201227882869
Epoch 6, Loss: 1.3936301380800447
Epoch 7, Loss: 1.3272724338381539
Epoch 8, Loss: 1.2574936494028945
Epoch 9, Loss: 1.227182417113452
Epoch 10, Loss: 1.1873740835191169
Koniec treningu
Model zapisany jako waste_model_2.pth


Poniżej jest kod do testowania na zbiorze /test. Żeby to działało trzeba odpalić 2 pierwsze fragmenty Code, czyli z funkcjami do ekstrakcji i do tworzenia, nie trzeba odpalać fragemntu z trenowaniem.

In [ ]:
import torch
import os
from torch.utils.data import DataLoader


classes = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
test_dir = 'Waste-Classification-1/test'
model_path = 'waste_model.pth'


class CustomDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.classes = sorted(os.listdir(root_dir))
        self.samples = []

        # Robimy listę wszystkich plików raz na początku
        for target_class in self.classes:
            class_dir = os.path.join(root_dir, target_class)
            if not os.path.isdir(class_dir):
                continue
            for img_name in os.listdir(class_dir):
                self.samples.append((
                    os.path.join(class_dir, img_name),
                    self.classes.index(target_class)
                ))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        features = przerabianko.extract_features(img_path)
        
        # Obsługa błędnych zdjęć (np. jeśli OpenCV nie wczyta pliku)
        if features is None:
            return torch.zeros(self.number_features), torch.tensor(label)
            
        return features, torch.tensor(label)
    

def evaluate():

    # 1. Przygotowanie danych testowych
    test_dataset = CustomDataset(root_dir=test_dir)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    correct = 0
    total = 0
    
    class_correct = {cls: 0 for cls in classes}
    class_total = {cls: 0 for cls in classes}

    # 2. Konfiguracja
    sample_features, _ = test_dataset[0]
    input_size = sample_features.shape[0] # Żeby był flex i nie trzeba by było liczyć tego co zmienioną/dodaną ceche cechę.
    
    # Nazwy klas - muszą być w tej samej kolejności co przy trenowaniu, ale to jest w json
    num_classes = len(classes)

    # 3. Wczytanie modelu
    net, _, _ = model_in(input_size=input_size, num_classes=num_classes)
    
    if not os.path.exists(model_path):
        print(f"Błąd: Nie znaleziono pliku modelu {model_path}!")
        return

    net.load_state_dict(torch.load(model_path))
    net.eval() # BARDZO WAŻNE: przełącza model w tryb testowania
    print("Model wczytany pomyślnie.")

    print(f"Test na {len(test_dataset)} obrazach")

    # 4. Pętla testująca
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = net(inputs)
            
            # Tutaj argmax po prostu
            _, predicted = torch.max(outputs.data, 1)
            
            total += 1
            label_idx = labels.item()
            pred_idx = predicted.item()

            if pred_idx == label_idx:
                correct += 1
                class_correct[classes[label_idx]] += 1
            
            class_total[classes[label_idx]] += 1

    # 5. Wyniki dla testowanego
    accuracy = 100 * correct / total
    print("\n" + "="*30)
    print(f"WYNIK OGÓLNY: {correct}/{total} ({accuracy:.2f}%)")
    print("="*30)
    
    print("\nSkuteczność dla poszczególnych klas:")
    for cls in classes:
        if class_total[cls] > 0:
            cls_acc = 100 * class_correct[cls] / class_total[cls]
            print(f"- {cls:10}: {class_correct[cls]}/{class_total[cls]} ({cls_acc:.2f}%)")
        else:
            print(f"- {cls:10}: Brak obrazów testowych")

if __name__ == "__main__":
    evaluate()


Model wczytany pomyślnie.
Rozpoczynam testowanie na 253 obrazach...

WYNIK OGÓLNY: 170/253 (67.19%)

Skuteczność dla poszczególnych klas:
- cardboard : 34/44 (77.27%)
- glass     : 33/47 (70.21%)
- metal     : 22/40 (55.00%)
- paper     : 46/64 (71.88%)
- plastic   : 26/44 (59.09%)
- trash     : 9/14 (64.29%)


Poniżej kod do testowania modelu dla pojedyńczego obrazu. Również w celu działania trzeba odpalić dwa pierwsze fragmenty kodu, na wejściu ścieżka do obrazu

In [15]:
import torch
import os

classes = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]


def test_single_photo(model_path, photo_path):

    if not os.path.exists(model_path) or not os.path.exists(photo_path):
        print(f"Error: Nie znaleziono pliku modelu {model_path}")
        return
    if not os.path.exists(photo_path):
        print(f"Error: Nie znaleziono pliku zdjęcia {photo_path}")
        return

    features = extract_features(photo_path)

    model, _, _ = model_in(input_size=features.shape[0], num_classes=len(classes))
    model.load_state_dict(torch.load(model_path))
    model.eval() # BARDZO WAŻNE: przełącza model w tryb testowania/predykcji

    features = features.unsqueeze(0) # Zamienia wymiary np. z (6,) na (1, 6). Bez tego jest błąd.

    outputs = model(features)

    # Tutaj argmax po prostu
    _, predicted = torch.max(outputs, 1)

    return classes[predicted.item()]
    
    

if __name__ == "__main__":

    photo_link = 'Waste-Classification-1/test/glass/glass1_jpg.rf.1e7df84c6223247aa22de933f1126425.jpg'
    model_link = 'waste_model.pth'

    print('Klasa według modelu dla zdjęcia to: ', test_single_photo(model_link, photo_link))

    
    

Klasa według modelu dla zdjęcia to:  glass
